# Snake Game AI using Reinforcement Learning with Deep Q-Learning

- Reinforcement Learning involves five components i.e Agent, Environment, Policy, Reward Function, and States.
- Easy explanation is: The Agent (our snake in this case) performs defined actions to achieve a goal (eat the apple) within it's environment, a reward is given to the agent every time it achieves the goal, and a penalty is given when the agent fails to achieve the goal. The policy is a strategy or set of rules that an agent uses to decide which action to take based on its current state in the environment. In this way the agent will learn by itself the best ways to maximize the number of rewards it gets and hence, getting better and better eating more apples without hitting walls or itself.

## Let's first Create a Playable Snake Game, Let's Call it Mayoka Snake

Play this game by running the cell below.


In [2]:
import pygame
import random
from enum import Enum
from collections import namedtuple

class Direction(Enum):
    RIGHT = 1
    LEFT = 2
    UP = 3
    DOWN = 4

Point = namedtuple("Point", "x, y")

pygame.display.init()
pygame.font.init()

font = pygame.font.Font(pygame.font.get_default_font(), 25)

WHITE = (255, 255, 255)
RED   = (200, 0, 0)
BLUE1 = (89, 136, 66)
BLUE2 = (101, 156, 73)
BLACK = (0, 0, 0)

BLOCK_SIZE = 20
SPEED = 10

class SnakeGame:
    def __init__(self, width=640, height=480, playerName="Player"):
        self.w = width
        self.h = height
        self.playerName = playerName

        self.display = pygame.display.set_mode((self.w, self.h))
        pygame.display.set_caption("Mayoka Snake")
        self.clock = pygame.time.Clock()

        # Load and scale head image to match BLOCK_SIZE
        self.head_img_orig = pygame.image.load("head-snake.png").convert_alpha()
        self.head_img_orig = pygame.transform.scale(self.head_img_orig, (BLOCK_SIZE, BLOCK_SIZE))

        # Load and scale apple image to match BLOCK_SIZE
        self.apple_img = pygame.image.load("apple.png").convert_alpha()
        self.apple_img = pygame.transform.scale(self.apple_img, (BLOCK_SIZE, BLOCK_SIZE))

        self.direction = Direction.RIGHT
        self.head = Point(self.w / 2, self.h / 2)
        self.snake = [
            self.head,
            Point(self.head.x - BLOCK_SIZE, self.head.y),
            Point(self.head.x - 2 * BLOCK_SIZE, self.head.y),
        ]
        self.score = 0
        self.food = None
        self.placeFood()

    def placeFood(self):
        x = random.randint(0, (self.w - BLOCK_SIZE) // BLOCK_SIZE) * BLOCK_SIZE
        y = random.randint(0, (self.h - BLOCK_SIZE) // BLOCK_SIZE) * BLOCK_SIZE
        self.food = Point(x, y)
        if self.food in self.snake:
            self.placeFood()

    def playStep(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                quit()

            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_LEFT and self.direction != Direction.RIGHT:
                    self.direction = Direction.LEFT
                elif event.key == pygame.K_RIGHT and self.direction != Direction.LEFT:
                    self.direction = Direction.RIGHT
                elif event.key == pygame.K_UP and self.direction != Direction.DOWN:
                    self.direction = Direction.UP
                elif event.key == pygame.K_DOWN and self.direction != Direction.UP:
                    self.direction = Direction.DOWN

        self.moveSnake(self.direction)

        if self.isCollision():
            return True, self.score

        if self.head == self.food:
            self.score += 1
            self.placeFood()
        else:
            self.snake.pop()

        self.updateUi()
        self.clock.tick(SPEED)

        return False, self.score

    def isCollision(self):
        if self.head.x >= self.w or self.head.x < 0:
            return True
        if self.head.y >= self.h or self.head.y < 0:
            return True
        if self.head in self.snake[1:]:
            return True
        return False

    def moveSnake(self, direction: Direction):
        x = self.head.x
        y = self.head.y

        if direction == Direction.RIGHT:
            x += BLOCK_SIZE
        elif direction == Direction.LEFT:
            x -= BLOCK_SIZE
        elif direction == Direction.DOWN:
            y += BLOCK_SIZE
        elif direction == Direction.UP:
            y -= BLOCK_SIZE

        self.head = Point(x, y)
        self.snake.insert(0, self.head)

    def getRotatedHeadImage(self):
        """Returns the correctly rotated head image based on direction."""
        if self.direction == Direction.DOWN:
            return self.head_img_orig  # No rotation needed
        elif self.direction == Direction.UP:
            return pygame.transform.rotate(self.head_img_orig, 180)
        elif self.direction == Direction.RIGHT:
            return pygame.transform.rotate(self.head_img_orig, 90)  # Rotate counterclockwise
        elif self.direction == Direction.LEFT:
            return pygame.transform.rotate(self.head_img_orig, -90)  # Rotate clockwise

    def updateUi(self):
        self.display.fill(BLACK)

        for p in self.snake[1:]:
            pygame.draw.rect(
                self.display, 
                BLUE1, 
                pygame.Rect(p.x, p.y, BLOCK_SIZE, BLOCK_SIZE)
            )
            pygame.draw.rect(
                self.display,
                BLUE2,
                pygame.Rect(p.x + 4, p.y + 4, 12, 12)
            )

        head = self.snake[0]
        rotated_head = self.getRotatedHeadImage()
        self.display.blit(rotated_head, (head.x, head.y))

        # Draw the apple image instead of a red block
        self.display.blit(self.apple_img, (self.food.x, self.food.y))

        scoreText = font.render(
            f"Score: {self.score}  Speed: {SPEED}  Player: {self.playerName}",
            True,
            WHITE
        )
        self.display.blit(scoreText, [0, 0])
        pygame.display.flip()

def playGame(playerName="Player 1"):
    game = SnakeGame(playerName=playerName)

    while True:
        gameOver, score = game.playStep()
        if gameOver:
            break

    print("Game Over, final score:", score)
    pygame.quit()

if __name__ == "__main__":
    playGame()


Game Over, final score: 5


## Now Let's Create the Game Environment To be Played by the Agent (The Snake: Mayoka) 

In [8]:
# GAME ENVIRONMENT
import pygame
import random
from enum import Enum
from collections import namedtuple
import numpy as np

class Direction(Enum):
    RIGHT = 1
    LEFT = 2
    UP = 3
    DOWN = 4

Point = namedtuple("Point", "x, y")

pygame.display.init()
pygame.font.init()

font = pygame.font.Font(pygame.font.get_default_font(), 25)

# Updated colors
WHITE = (255, 255, 255)
RED = (200, 0, 0)
BLUE1 = (89, 136, 66)  # Updated snake body color
BLUE2 = (101, 156, 73)  # Updated snake body color
BLACK = (0, 0, 0)

BLOCK_SIZE = 20
SPEED = 120

class MayokaSnakeAI:
    def __init__(self, width=640, height=480, playerName="Mayoka"):
        self.w = width
        self.h = height
        self.playerName = playerName

        # Initialize display
        self.display = pygame.display.set_mode((self.w, self.h))
        pygame.display.set_caption("Mayoka Snake")
        self.clock = pygame.time.Clock()

        # Load and scale head image
        self.head_img_orig = pygame.image.load("head-snake.png").convert_alpha()
        self.head_img_orig = pygame.transform.scale(self.head_img_orig, (BLOCK_SIZE, BLOCK_SIZE))

        # Load and scale apple image
        self.apple_img = pygame.image.load("apple.png").convert_alpha()
        self.apple_img = pygame.transform.scale(self.apple_img, (BLOCK_SIZE, BLOCK_SIZE))

        # Initialize game
        self.reset()

    def reset(self):
        self.direction = Direction.RIGHT
        self.head = Point(self.w / 2, self.h / 2)
        self.snake = [
            self.head,
            Point(self.head.x - BLOCK_SIZE, self.head.y),
            Point(self.head.x - (2 * BLOCK_SIZE), self.head.y)
        ]
        self.score = 0
        self.food = None
        self.placeFood()
        self.frameIteration = 0

    def placeFood(self):
        x = random.randint(0, (self.w - BLOCK_SIZE) // BLOCK_SIZE) * BLOCK_SIZE
        y = random.randint(0, (self.h - BLOCK_SIZE) // BLOCK_SIZE) * BLOCK_SIZE
        self.food = Point(x, y)
        if self.food in self.snake:
            self.placeFood()

    def playStep(self, action):
        self.frameIteration += 1

        for event in pygame.event.get():
            if event == pygame.QUIT:
                pygame.quit()
                quit()

        # Move the snake
        self.moveSnake(action)

        # Reward logic
        reward = 0

        if self.isCollision() or self.frameIteration > 100 * len(self.snake):
            gameOver = True
            reward -= 10
            return reward, gameOver, self.score

        if self.head == self.food:
            self.score += 1
            reward = 10
            self.placeFood()
        else:
            self.snake.pop()

        # Update UI and clock
        self.updateUi()
        self.clock.tick(SPEED)
        
        return reward, False, self.score

    def isCollision(self, p: Point = None):
        if p is None:
            p = self.head

        # Check if it hits border
        if p.x >= self.w or p.x < 0 or p.y >= self.h or p.y < 0:
            return True

        # Check if it hits itself
        if p in self.snake[1:]:
            return True

        return False

    def moveSnake(self, action):
        # Actions -> [straight, right, left]
        clockWiseDirections = [Direction.RIGHT, Direction.DOWN, Direction.LEFT, Direction.UP]
        currentDirectionIndex = clockWiseDirections.index(self.direction)

        newDirection = self.direction

        if np.array_equal(action, [0, 1, 0]):  # Turn right
            newDirection = clockWiseDirections[(currentDirectionIndex + 1) % 4]
        elif np.array_equal(action, [0, 0, 1]):  # Turn left
            newDirection = clockWiseDirections[(currentDirectionIndex - 1) % 4]

        self.direction = newDirection

        x = self.head.x
        y = self.head.y

        if self.direction == Direction.RIGHT:
            x += BLOCK_SIZE
        elif self.direction == Direction.LEFT:
            x -= BLOCK_SIZE
        elif self.direction == Direction.DOWN:
            y += BLOCK_SIZE
        elif self.direction == Direction.UP:
            y -= BLOCK_SIZE

        self.head = Point(x, y)
        self.snake.insert(0, self.head)

    def getRotatedHeadImage(self):
        """Rotate the head image based on direction."""
        if self.direction == Direction.DOWN:
            return self.head_img_orig  # No rotation needed
        elif self.direction == Direction.UP:
            return pygame.transform.rotate(self.head_img_orig, 180)
        elif self.direction == Direction.RIGHT:
            return pygame.transform.rotate(self.head_img_orig, -90)  # Rotate counterclockwise
        elif self.direction == Direction.LEFT:
            return pygame.transform.rotate(self.head_img_orig, 90)  # Rotate clockwise

    def updateUi(self):
        self.display.fill(BLACK)

        # Draw snake body
        for p in self.snake[1:]:
            pygame.draw.rect(self.display, BLUE1, pygame.Rect(p.x, p.y, BLOCK_SIZE, BLOCK_SIZE))
            pygame.draw.rect(self.display, BLUE2, pygame.Rect(p.x + 4, p.y + 4, 12, 12))

        # Draw the snake head with the rotated image
        rotated_head = self.getRotatedHeadImage()
        self.display.blit(rotated_head, (self.head.x, self.head.y))

        # Draw the apple
        self.display.blit(self.apple_img, (self.food.x, self.food.y))

        # Display score
        scoreText = font.render(f"Score: {self.score} Speed: {SPEED} Player: {self.playerName}", True, WHITE)
        self.display.blit(scoreText, [0, 0])

        pygame.display.flip()

    def setPlayerName(self, name):
        self.playerName = name


## The MODEL

In [9]:
# THE MODEL
# A simple neural network, having 11 input neurons (the size of our state array) and 3 output neurons (the number of our potential moves straight, right and left)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class LinearQNet(nn.Module):

    def __init__(self, inputSize, hiddenSize, outputSize):
        super().__init__()

        self.linear1 = nn.Linear(inputSize, hiddenSize)
        self.linear2 = nn.Linear(hiddenSize, outputSize)

    def forward(self, X):
        out = self.linear1(X)
        out = F.relu(out)
        out = self.linear2(out)

        return out

## The Trainer

In [10]:
# THE TRAINER
# This will be used to actually train our model based on the states, actions and the rewards provided by the agent.

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class QTrainner:

    def __init__(self, model, lr, gamma):
        self.model = model
        self.lr = lr
        self.gamma = gamma

        self.optimizer = optim.Adam(model.parameters(), self.lr)
        self.lossFunction = nn.MSELoss()

    def trainStep(self, state, action, reward, newState, done):

        stateTensor = torch.tensor(state, dtype=torch.float)
        actionTensor = torch.tensor(action, dtype=torch.long)
        rewardTensor = torch.tensor(reward, dtype=torch.float)
        newStateTensor = torch.tensor(newState, dtype=torch.float)

        if len(stateTensor.shape) == 1:
            stateTensor = torch.unsqueeze(stateTensor, 0)
            newStateTensor = torch.unsqueeze(newStateTensor, 0)
            actionTensor = torch.unsqueeze(actionTensor, 0)
            rewardTensor = torch.unsqueeze(rewardTensor, 0)
            done = (done, )

        # 1. predicted q values with current state
        prediction = self.model(stateTensor)

        # Q_new = 2. reward + gamma * max(next predicted q value) -> only do this if not done
        target = prediction.clone()

        for i in range(len(done)):
            Qnew = rewardTensor[i]

            if not done[i]:
                Qnew = rewardTensor[i] + self.gamma * torch.max(self.model(newStateTensor[i]))

            target[i][torch.argmax(actionTensor).item()] = Qnew


        self.optimizer.zero_grad()
        loss = self.lossFunction(target, prediction)
        loss.backward()

        self.optimizer.step()

## The Agent

In [11]:
# THE AGENT

# agent does the learning and performs the actions

import torch
import random
import numpy as np
from collections import deque

MAX_MEMORY = 100_000
BATCH_SIZE = 1000
LR = 0.001

class Agent:
    def __init__(self):
        self.numberOfGames = 0
        # controls randomness
        self.epsilon = 0
        # discount rate
        self.gamma = 0
        # the last actions, if we reach the limit we will remove the oldest
        self.memory = deque(maxlen=MAX_MEMORY)

        # the modelneeds to have 11 inputs, as our world status grid has 11 elements and the output is 3, as we have 3 directions
        self.model = LinearQNet(11, 256, 3)
        self.trainner = QTrainner(self.model, lr=LR, gamma=self.gamma)

    # this gives us the state of the world
    def getState(self, game):
        head = game.head

        point_left = Point(head.x - BLOCK_SIZE, head.y)
        point_right = Point(head.x + BLOCK_SIZE, head.y)
        point_up = Point(head.x, head.y - BLOCK_SIZE)
        point_down = Point(head.x, head.y + BLOCK_SIZE)

        direction_left = game.direction == Direction.LEFT
        direction_right = game.direction == Direction.RIGHT
        direction_up = game.direction == Direction.UP
        direction_down = game.direction == Direction.DOWN

        state = [
            # Danger straight
            (direction_right and game.isCollision(point_right)) or
            (direction_left and game.isCollision(point_left)) or
            (direction_up and game.isCollision(point_up)) or
            (direction_down and game.isCollision(point_down)),

            # Danger right
            (direction_up and game.isCollision(point_right)) or
            (direction_down and game.isCollision(point_left)) or
            (direction_left and game.isCollision(point_up)) or
            (direction_right and game.isCollision(point_down)),

            # Danger left
            (direction_down and game.isCollision(point_right)) or
            (direction_up and game.isCollision(point_left)) or
            (direction_right and game.isCollision(point_up)) or
            (direction_left and game.isCollision(point_down)),

            # Move direction
            direction_left,
            direction_right,
            direction_up,
            direction_down,

            # Food location
            game.food.x < game.head.x,  # food left
            game.food.x > game.head.x,  # food right
            game.food.y < game.head.y,  # food up
            game.food.y > game.head.y  # food down
        ]

        return np.array(state, dtype=int)

    def remember(self, state, action, reward, nextState, done):
        self.memory.append((state, action, reward, nextState, done))

    def trainLongMemory(self):
        if len(self.memory) < BATCH_SIZE:
            sample = self.memory
        else:
            sample = random.sample(self.memory, BATCH_SIZE)

        states, actions, rewards, nextStates, dones = zip(*sample)
        self.trainner.trainStep(states, actions, rewards, nextStates, dones)

    def trainShortMemory(self, state, action, reward, nextState, done):
        self.trainner.trainStep(state, action, reward, nextState, done)

    def getAction(self, state):
        # in the beginning will do some random moves, tradeoff between exploration and exploataition
        self.epsilon = 80 - self.numberOfGames
        finalMove = [0, 0, 0]

        if random.randint(0, 200) < self.epsilon:
            move = random.randint(0, 2)
            finalMove[move] = 1
        else:
            stateTensor = torch.tensor(state, dtype=torch.float)
            prediction = self.model(stateTensor)
            move = torch.argmax(prediction).item()
            finalMove[move] = 1

        return finalMove

## Start Training

In [12]:
# Training and letting AI play the game
# %%time

scoresHistory = []
meanScores = []

def train():
    
    totalScore = 0
    bestScore = 0

    agent = Agent()
    game = MayokaSnakeAI()

    # we train the model for 200 games
    while agent.numberOfGames < 200:

        game.setPlayerName("Machine epoch " + str(agent.numberOfGames))

        # get old state
        oldState = agent.getState(game)

        # move
        finalMove = agent.getAction(oldState)

        # perform move and get new state
        reward, done, score = game.playStep(finalMove)

        newState = agent.getState(game)

        # train short memory
        agent.trainShortMemory(oldState, finalMove, reward, newState, done)

        # remember
        agent.remember(oldState, finalMove, reward, newState, done)

        if done:
            # train long memory
            game.reset()
            agent.numberOfGames += 1
            agent.trainLongMemory()

            if score > bestScore:
                bestScore = score

            totalScore += score
            meanScore = (totalScore / agent.numberOfGames)

            scoresHistory.append(score)
            meanScores.append(meanScore)

            if score == bestScore or agent.numberOfGames % 10 == 0:
                print("Game number: ", agent.numberOfGames, "Score: ", score, "Best Score: ", bestScore, "Mean scores: ", meanScore)


# run the trainning method                
train()

Game number:  1 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  2 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  3 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  4 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  5 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  6 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  7 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  8 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  9 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  10 Score:  0 Best Score:  0 Mean scores:  0.0
Game number:  11 Score:  1 Best Score:  1 Mean scores:  0.09090909090909091
Game number:  13 Score:  1 Best Score:  1 Mean scores:  0.15384615384615385


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.plot(scoresHistory)
plt.plot(meanScores)
plt.legend(["Score", "Mean Score"])